# GhostExec — Colab reward demo

Set the Colab working directory to the **ghostexec** package root (the folder that contains `pyproject.toml`), or clone your public repo to `/content/ghostexec`. The next cell installs the package in editable mode.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd() / "ghostexec", Path("/content/ghostexec")]
root = None
for p in candidates:
    if (p / "pyproject.toml").exists():
        root = p
        break
if root is None:
    print("Could not find ghostexec/pyproject.toml; skipping editable install.")
else:
    os.chdir(root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=False)


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
train_py = ROOT / "training" / "train.py"
if not train_py.is_file():
    raise RuntimeError("training/train.py not found; cwd should be the ghostexec package root.")

log_path = ROOT / "outputs" / "training" / "colab_episode_returns.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)
if log_path.exists():
    log_path.unlink()

subprocess.check_call(
    [
        sys.executable,
        str(train_py),
        "--backend",
        "local",
        "--agent",
        "smart",
        "--episodes",
        "28",
        "--max-steps",
        "12",
        "--log-path",
        str(log_path),
        "--checkpoint-dir",
        str(ROOT / "outputs" / "training" / "colab_ckpt"),
    ],
    cwd=str(ROOT),
)

rows = [json.loads(ln) for ln in log_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
summary_path = ROOT / "outputs" / "training" / "colab_ckpt" / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

try:
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    import matplotlib.pyplot as plt

plt.figure(figsize=(9, 3.5))
plt.plot([r["episode"] for r in rows], [r["return"] for r in rows], marker="o", linewidth=1)
plt.xlabel("Episode index")
plt.ylabel("Episode return (sum of step rewards)")
plt.title("GhostExec — smart policy episode returns (demo)")
plt.grid(True, alpha=0.3)
plt.show()

print("First episode first action:", json.dumps(summary.get("first_episode_first_action"), indent=2))
print("Last episode first action:", json.dumps(summary.get("last_episode_first_action"), indent=2))


## What the curve shows

Each point is one **full episode** (up to 12 steps) using the bundled **smart** scripted policy.

For a full **LLM + PPO/GRPO** loop, install optional dependencies (`uv sync --extra training`) and extend `training/train.py` with Hugging Face TRL against the same JSONL log format.
